# Certificación Profesional en Desarrollador de Agentes de IA
## Clase 9 — Herramientas y workflows agénticos

ITBA Educación Ejecutiva

Automatización inteligente bajo control: hasta ahora el sistema responde o dice "no sé". Hoy aprende a **actuar** — y a hacerlo con límites, validación y trazabilidad.

### Cómo se usa este notebook

- El docente corre y explica cada celda en vivo. No programás en simultáneo durante la clase.
- Celdas **🏠 Practicá**: tarea individual para después de la clase.
- **Sección 10 (Laboratorio)**: checkpoint del proyecto integrador.

## Objetivo de la clase

Al final de la clase vas a poder:
- Distinguir cuándo un chatbot o un workflow alcanzan, y cuándo hace falta un agente — y cuándo ninguno de los tres es la respuesta correcta.
- Diseñar herramientas con contratos de entrada/salida validados, no solo "funciones que el LLM puede llamar" — y reconocer el mismo patrón detrás de la sintaxis específica de cada proveedor.
- Construir un workflow con estado que decide entre responder con RAG o ejecutar una acción externa.
- Aplicar límites de pasos, prevención de loops, reintentos, fallbacks e idempotencia a ese workflow.
- Registrar cada decisión y cada acción ejecutada, de forma auditable.

**Checkpoint del proyecto integrador de hoy**: sistema de tickets en SQLite funcionando, con `consultar_estado_ticket` integrada por tool calling (ver `proyecto-integrador/README.md`, checkpoint Clase 9).

## Agenda

1. Chatbot, workflow o agente — y cuándo no conviene ninguno
2. Tool calling: diseño de herramientas y contratos
3. Tool calling en otros proveedores (OpenAI, Anthropic)
4. Validación de argumentos
5. Estado de ejecución y rutas condicionales
6. Límites de pasos y prevención de loops
7. Reintentos y fallbacks
8. Idempotencia
9. Registro de decisiones y acciones
10. Síntesis: el workflow completo
11. Laboratorio: sistema de tickets integrado

## 0. Punto de partida

Reconstruimos el RAG de las Clases 6-9 (mismo corpus de juguete) — hoy le vamos a agregar la capacidad de actuar, no solo responder.

In [ ]:
import os, json, sqlite3, time
from datetime import datetime
from dotenv import load_dotenv
load_dotenv()

import boto3
bedrock = boto3.client("bedrock-runtime", region_name=os.environ.get("AWS_REGION", "us-east-1"))
MODEL_ID = os.environ.get("BEDROCK_MODEL_ID_CLAUDE", "global.anthropic.claude-sonnet-4-6")
EMBED_MODEL_ID = os.environ.get("BEDROCK_MODEL_ID_EMBED", "cohere.embed-v4:0")

def embed_bedrock(textos, input_type="search_document"):
    body = json.dumps({"texts": textos, "input_type": input_type, "embedding_types": ["float"]})
    resp = bedrock.invoke_model(modelId=EMBED_MODEL_ID, body=body, accept="*/*", contentType="application/json")
    return json.loads(resp["body"].read())["embeddings"]["float"]

In [ ]:
import chromadb

corpus = [
    {"id": "apis-001", "texto": "La Converse API de Bedrock unifica la forma de llamar a distintos modelos: cambia el modelId, no el código.", "doc": "guia-apis"},
    {"id": "apis-002", "texto": "Los parámetros de inferencia como temperature y maxTokens van en inferenceConfig en Bedrock.", "doc": "guia-apis"},
    {"id": "seguridad-001", "texto": "La inyección de prompts ocurre cuando contenido no confiable manipula las instrucciones del modelo.", "doc": "guia-seguridad"},
    {"id": "rag-001", "texto": "Un pipeline RAG combina recuperación de evidencia con generación sustentada en esa evidencia.", "doc": "guia-rag"},
]

cliente_chroma = chromadb.Client()
coleccion = cliente_chroma.get_or_create_collection("corpus_clase9")
coleccion.add(
    ids=[c["id"] for c in corpus],
    embeddings=embed_bedrock([c["texto"] for c in corpus]),
    documents=[c["texto"] for c in corpus],
    metadatas=[{"doc": c["doc"]} for c in corpus],
)

def recuperar(consulta, k=3, umbral=1.0):
    vec = embed_bedrock([consulta], input_type="search_query")[0]
    r = coleccion.query(query_embeddings=[vec], n_results=k)
    chunks = [{"chunk_id": i, "texto": t, "distancia": d}
              for i, t, d in zip(r["ids"][0], r["documents"][0], r["distances"][0])]
    return [c for c in chunks if c["distancia"] < umbral]

print("Corpus listo:", coleccion.count(), "chunks")

## 1. Chatbot, workflow o agente — y cuándo no conviene ninguno

| | Chatbot | Workflow | Agente |
|---|---|---|---|
| Control del flujo | Ninguno (una respuesta por mensaje) | Fijo, definido por el desarrollador | El modelo decide el próximo paso |
| Previsibilidad | Alta | Alta | Más baja — puede sorprender |
| Casos de uso | FAQ simple, sin acciones | Pasos conocidos de antemano (ej. siempre: recuperar → generar) | La secuencia de pasos depende de la consulta |
| Costo de desarrollo/debugging | Bajo | Medio | Alto |

El sistema de tickets de hoy es, en rigor, un **workflow con un punto de decisión** (¿hay evidencia o no?), no un agente libre — la ruta posible está definida de antemano, el modelo no decide "inventar" una herramienta nueva. Esta distinción importa: cuanta más libertad le das al modelo sobre qué hacer, menos previsible y más difícil de debuggear es el sistema.

**Cuándo NO conviene usar un agente** (ni siquiera un workflow): cuando la tarea es simple y determinística (un chatbot de FAQ alcanza), cuando la previsibilidad es crítica (un error de interpretación del modelo sobre qué paso seguir tiene consecuencias serias), o cuando ya existe una integración simple y directa que no necesita "inteligencia" para decidir el flujo — agregar un LLM a algo que ya funciona con un if/else es complejidad injustificada.

## 2. Tool calling: diseño de herramientas y contratos

Una herramienta no es "una función de Python que el modelo puede llamar" — es un **contrato**: nombre, descripción, schema de entrada, y una garantía de qué hace y qué devuelve. El modelo nunca ejecuta la función directamente (Clase 3) — decide *cuándo* y *con qué argumentos* invocarla; el código la ejecuta.

Hoy construimos el sistema de tickets: una base SQLite y una herramienta de consulta.

In [ ]:
conexion_tickets = sqlite3.connect("tickets.db")
conexion_tickets.execute('''
    CREATE TABLE IF NOT EXISTS tickets (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        pregunta TEXT NOT NULL,
        contexto_recuperado TEXT,
        estado TEXT NOT NULL DEFAULT 'abierto',
        motivo_escalamiento TEXT,
        fecha_creacion TEXT NOT NULL
    )
''')
conexion_tickets.execute('''
    CREATE TABLE IF NOT EXISTS registro_acciones (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        timestamp TEXT NOT NULL,
        consulta_origen TEXT,
        decision TEXT,
        herramienta TEXT,
        argumentos TEXT,
        resultado TEXT
    )
''')
conexion_tickets.commit()
print("Base de tickets lista.")

In [ ]:
def consultar_estado_ticket(ticket_id: int) -> dict:
    """Contrato: recibe un ticket_id (int), devuelve {encontrado, estado, pregunta} o {encontrado: False}."""
    fila = conexion_tickets.execute(
        "SELECT estado, pregunta FROM tickets WHERE id = ?", (ticket_id,)
    ).fetchone()
    if not fila:
        return {"encontrado": False}
    return {"encontrado": True, "estado": fila[0], "pregunta": fila[1]}


TOOL_CONFIG = {
    "tools": [{
        "toolSpec": {
            "name": "consultar_estado_ticket",
            "description": "Consulta el estado actual de un ticket de soporte por su ID numérico.",
            "inputSchema": {"json": {
                "type": "object",
                "properties": {"ticket_id": {"type": "integer", "description": "ID numérico del ticket"}},
                "required": ["ticket_id"],
            }},
        }
    }]
}

**🏠 Practicá**: escribí el docstring de `consultar_estado_ticket` como si fuera para alguien que nunca vio el código — la descripción del `toolSpec` es, en la práctica, la única documentación que tiene el modelo para decidir cuándo usar la herramienta. Una descripción ambigua ("consulta un ticket") genera invocaciones incorrectas con más frecuencia que una específica.

## 3. Tool calling en otros proveedores

El contrato (`toolSpec`/`input_schema`, validar antes de ejecutar) es el mismo concepto en todos los proveedores — pero cada SDK lo expone con una forma distinta. Si tu proveedor no es Bedrock, esto es lo que cambia.

In [ ]:
# --- OpenAI (Chat Completions API) ---
from openai import OpenAI

def tool_calling_openai(consulta: str):
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    tools_openai = [{
        "type": "function",
        "function": {
            "name": "consultar_estado_ticket",
            "description": "Consulta el estado actual de un ticket de soporte por su ID numérico.",
            "parameters": {
                "type": "object",
                "properties": {"ticket_id": {"type": "integer", "description": "ID numérico del ticket"}},
                "required": ["ticket_id"],
            },
        },
    }]

    completion = client.chat.completions.create(
        model=os.environ.get("OPENAI_MODEL_ID", "gpt-4o"),
        messages=[{"role": "user", "content": consulta}],
        tools=tools_openai,
    )
    message = completion.choices[0].message

    if message.tool_calls:
        tool_call = message.tool_calls[0]
        args = json.loads(tool_call.function.arguments)
        resultado = ejecutar_herramienta_validada(tool_call.function.name, args)

        followup = client.chat.completions.create(
            model=os.environ.get("OPENAI_MODEL_ID", "gpt-4o"),
            messages=[
                {"role": "user", "content": consulta},
                message,
                {"role": "tool", "tool_call_id": tool_call.id, "content": json.dumps(resultado)},
            ],
        )
        return followup.choices[0].message.content
    return message.content

# tool_calling_openai("¿Cuál es el estado del ticket 1?")  # descomentar si tenés OPENAI_API_KEY configurada

In [ ]:
# --- Anthropic directo (Messages API) ---
import anthropic as anthropic_sdk

def tool_calling_anthropic(consulta: str):
    client = anthropic_sdk.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    tools_anthropic = [{
        "name": "consultar_estado_ticket",
        "description": "Consulta el estado actual de un ticket de soporte por su ID numérico.",
        "input_schema": {
            "type": "object",
            "properties": {"ticket_id": {"type": "integer", "description": "ID numérico del ticket"}},
            "required": ["ticket_id"],
        },
    }]

    response = client.messages.create(
        model=os.environ.get("ANTHROPIC_MODEL_ID", "claude-sonnet-4-6"),
        max_tokens=500,
        tools=tools_anthropic,
        messages=[{"role": "user", "content": consulta}],
    )

    if response.stop_reason == "tool_use":
        bloque_tool = next(b for b in response.content if b.type == "tool_use")
        resultado = ejecutar_herramienta_validada(bloque_tool.name, bloque_tool.input)

        followup = client.messages.create(
            model=os.environ.get("ANTHROPIC_MODEL_ID", "claude-sonnet-4-6"),
            max_tokens=500,
            tools=tools_anthropic,
            messages=[
                {"role": "user", "content": consulta},
                {"role": "assistant", "content": response.content},
                {"role": "user", "content": [
                    {"type": "tool_result", "tool_use_id": bloque_tool.id, "content": json.dumps(resultado)}
                ]},
            ],
        )
        return followup.content[0].text
    return response.content[0].text

# tool_calling_anthropic("¿Cuál es el estado del ticket 1?")  # descomentar si tenés ANTHROPIC_API_KEY configurada

Notá el patrón idéntico en los tres proveedores (Bedrock, OpenAI, Anthropic): (1) declarar el schema de la herramienta, (2) revisar si la respuesta pide usarla, (3) ejecutar **vos** la función ya validada, (4) mandar el resultado de vuelta en un segundo llamado. Lo único que cambia es la forma exacta del JSON — exactamente el tipo de diferencia que la abstracción de proveedores (Clase 3, `LLMProvider`) existe para esconder.

**🏠 Practicá**: si tu proveedor del TP es Google o Ollama, buscá en su documentación cómo se llama este mismo patrón (`function calling` en Gemini, soporte más limitado/variable en modelos vía Ollama según el modelo) y escribí el equivalente de `tool_calling_openai` para tu caso.

## 4. Validación de argumentos

El modelo puede, en teoría, pasar cualquier cosa como argumento — un `ticket_id` que no es un número, un string vacío, un valor fuera de rango. **Nunca ejecutes una herramienta con los argumentos del modelo sin validarlos primero**, igual que no confiarías en el input de un usuario sin validar.

In [ ]:
from pydantic import BaseModel, ValidationError, field_validator

class ArgsConsultarTicket(BaseModel):
    ticket_id: int

    @field_validator("ticket_id")
    @classmethod
    def debe_ser_positivo(cls, v):
        if v <= 0:
            raise ValueError("ticket_id debe ser un entero positivo")
        return v


def ejecutar_herramienta_validada(nombre: str, argumentos_crudos: dict) -> dict:
    if nombre == "consultar_estado_ticket":
        try:
            args = ArgsConsultarTicket(**argumentos_crudos)
        except ValidationError as e:
            return {"error": f"Argumentos inválidos: {e}"}
        return consultar_estado_ticket(args.ticket_id)
    return {"error": f"Herramienta desconocida: {nombre}"}


print(ejecutar_herramienta_validada("consultar_estado_ticket", {"ticket_id": -5}))

**🏠 Practicá**: probá `ejecutar_herramienta_validada` con `{"ticket_id": "no-es-un-numero"}` y con `{}` (sin el argumento) — confirmá que ninguno de los dos casos llega a tocar la base de datos sin un `ticket_id` válido.

## 5. Estado de ejecución y rutas condicionales

El workflow de hoy tiene exactamente un punto de decisión explícito: **¿el RAG encontró evidencia suficiente?** Esa pregunta define qué ruta se ejecuta — no la decide el modelo libremente, la decide el código, basado en el resultado de la recuperación. Esto es lo que lo hace un *workflow*, no un agente totalmente libre (sección 1).

In [ ]:
def crear_ticket(pregunta: str, contexto_recuperado: str = "") -> int:
    cursor = conexion_tickets.execute(
        "INSERT INTO tickets (pregunta, contexto_recuperado, fecha_creacion) VALUES (?, ?, ?)",
        (pregunta, contexto_recuperado, datetime.now().isoformat()),
    )
    conexion_tickets.commit()
    return cursor.lastrowid


def workflow_paso(consulta: str) -> dict:
    chunks = recuperar(consulta)

    if chunks:  # RUTA A: hay evidencia -> responder con RAG (y la herramienta de tickets disponible por si la pregunta la necesita)
        contexto = "\n\n".join(f"[{i+1}] {c['texto']}" for i, c in enumerate(chunks))
        prompt = f"<evidencia>\n{contexto}\n</evidencia>\n\n<pregunta>\n{consulta}\n</pregunta>"
        respuesta = bedrock.converse(
            modelId=MODEL_ID,
            messages=[{"role": "user", "content": [{"text": prompt}]}],
            toolConfig=TOOL_CONFIG,
        )
        if respuesta["stopReason"] == "tool_use":
            # el modelo decidió que además necesita consultar un ticket -> se resuelve en la sección 2-3
            bloque_tool = next(b["toolUse"] for b in respuesta["output"]["message"]["content"] if "toolUse" in b)
            resultado_tool = ejecutar_herramienta_validada(bloque_tool["name"], bloque_tool["input"])
            return {"ruta": "RAG+herramienta", "resultado": resultado_tool}
        texto = respuesta["output"]["message"]["content"][0]["text"]
        return {"ruta": "RAG", "resultado": texto}

    else:  # RUTA B: sin evidencia suficiente -> acción determinística, no generación
        ticket_id = crear_ticket(consulta)
        return {"ruta": "ticket_creado", "resultado": f"No encontré evidencia suficiente. Abrí el ticket #{ticket_id} para que alguien del equipo lo revise."}


print(workflow_paso("¿Qué parámetro controla la temperatura de generación?"))
print(workflow_paso("¿Cómo configuro un balanceador de carga para mi base de datos?"))

Notá que la ruta B **no llama al modelo generativo en absoluto** — es una acción de código puro. Esto es intencional: si no hay evidencia, no tiene sentido arriesgar una alucinación generando una respuesta de todas formas; la acción correcta es determinística.

## 6. Límites de pasos y prevención de loops

`workflow_paso` de hoy hace como máximo 2 llamadas al modelo (la inicial, y una de seguimiento si hubo `tool_use`). En un agente más libre (varias herramientas, el modelo decide encadenar varias llamadas), hace falta un límite explícito — sin él, un modelo que "decide" seguir llamando herramientas puede entrar en un loop que no termina nunca (y que cuesta dinero en cada vuelta).

In [ ]:
def workflow_con_limite(consulta: str, max_pasos: int = 3) -> dict:
    pasos_dados = 0
    messages = [{"role": "user", "content": [{"text": consulta}]}]

    while pasos_dados < max_pasos:
        pasos_dados += 1
        resp = bedrock.converse(modelId=MODEL_ID, messages=messages, toolConfig=TOOL_CONFIG)
        messages.append(resp["output"]["message"])

        if resp["stopReason"] != "tool_use":
            return {"resultado": resp["output"]["message"]["content"][0]["text"], "pasos": pasos_dados}

        bloque_tool = next(b["toolUse"] for b in resp["output"]["message"]["content"] if "toolUse" in b)
        resultado_tool = ejecutar_herramienta_validada(bloque_tool["name"], bloque_tool["input"])
        messages.append({"role": "user", "content": [{"toolResult": {
            "toolUseId": bloque_tool["toolUseId"], "content": [{"json": resultado_tool}]
        }}]})

    return {"resultado": "Se alcanzó el límite de pasos sin una respuesta final.", "pasos": pasos_dados}


print(workflow_con_limite("¿Cuál es el estado del ticket 1?"))

**🏠 Practicá**: bajá `max_pasos` a 1 y corré una consulta que dispare una herramienta — confirmá que el workflow corta con el mensaje de "límite de pasos alcanzado" en vez de fallar con un error críptico o colgarse.

## 7. Reintentos y fallbacks

Igual que en la Clase 2-3, pero ahora aplicado a cada llamada del workflow: si `bedrock.converse` falla por throttling, hay que reintentar con backoff; si falla con un error no reintentable, hace falta un **fallback** — una respuesta degradada pero útil, no un error sin manejar.

In [ ]:
from botocore.exceptions import ClientError
import random

def converse_con_reintentos(**kwargs):
    for intento in range(3):
        try:
            return bedrock.converse(**kwargs)
        except ClientError as err:
            codigo = err.response["Error"]["Code"]
            if codigo == "ThrottlingException" and intento < 2:
                time.sleep((2 ** intento) + random.uniform(0, 1))
                continue
            raise

def workflow_con_fallback(consulta: str) -> dict:
    try:
        return workflow_paso(consulta)
    except Exception as e:
        # Fallback: ante cualquier error no manejado, igual abrimos un ticket -- nunca dejar la consulta sin registro
        ticket_id = crear_ticket(consulta, contexto_recuperado=f"[fallback por error: {e}]")
        return {"ruta": "fallback_error", "resultado": f"Hubo un problema técnico. Abrí el ticket #{ticket_id}."}

print(workflow_con_fallback("¿Qué es RAG?"))

## 8. Idempotencia

Si el mismo usuario pregunta lo mismo dos veces (por un reintento del cliente, un doble clic, o un reintento automático tras un timeout), `crear_ticket` tal como está hoy crea **dos tickets duplicados**. Una operación idempotente produce el mismo resultado final sin importar cuántas veces se ejecute con el mismo input.

In [ ]:
def crear_ticket_idempotente(pregunta: str, contexto_recuperado: str = "") -> int:
    existente = conexion_tickets.execute(
        "SELECT id FROM tickets WHERE pregunta = ? AND estado = 'abierto'", (pregunta,)
    ).fetchone()
    if existente:
        return existente[0]  # ya existe un ticket abierto para esta misma pregunta -> no duplicar
    return crear_ticket(pregunta, contexto_recuperado)


id_1 = crear_ticket_idempotente("¿Cómo configuro un balanceador de carga?")
id_2 = crear_ticket_idempotente("¿Cómo configuro un balanceador de carga?")
print(f"Primer llamado: ticket #{id_1}  |  Segundo llamado: ticket #{id_2}  (¿mismo ticket? {id_1 == id_2})")

**🏠 Practicá**: esta implementación de idempotencia es ingenua — compara el texto exacto de la pregunta. ¿Qué pasa si alguien pregunta lo mismo con otras palabras? Pensá (no hace falta implementarlo) cómo usarías embeddings (Clase 4-5) para detectar "preguntas casi iguales" en vez de solo "preguntas idénticas".

## 9. Registro de decisiones y acciones

Para poder auditar el sistema después (¿por qué se abrió este ticket? ¿qué decisión tomó el workflow y cuándo?), cada paso se registra — no solo el resultado final que ve el usuario.

In [ ]:
def registrar_accion(consulta_origen: str, decision: str, herramienta: str = None, argumentos: dict = None, resultado=None):
    conexion_tickets.execute(
        "INSERT INTO registro_acciones (timestamp, consulta_origen, decision, herramienta, argumentos, resultado) "
        "VALUES (?, ?, ?, ?, ?, ?)",
        (datetime.now().isoformat(), consulta_origen, decision, herramienta,
         json.dumps(argumentos) if argumentos else None, json.dumps(resultado, default=str) if resultado else None),
    )
    conexion_tickets.commit()


def workflow_con_registro(consulta: str) -> dict:
    resultado = workflow_con_fallback(consulta)
    registrar_accion(consulta, decision=resultado["ruta"], resultado=resultado["resultado"])
    return resultado


workflow_con_registro("¿Qué controla la temperatura de generación?")
workflow_con_registro("¿Cómo migro una base de datos Oracle a PostgreSQL?")

for fila in conexion_tickets.execute("SELECT consulta_origen, decision FROM registro_acciones ORDER BY id DESC LIMIT 5"):
    print(fila)

## 10. Síntesis: el workflow completo

Todas las piezas de hoy en una sola función: contrato validado, ruta condicional según estado, límite de pasos, reintentos con fallback, idempotencia y registro.

In [ ]:
def workflow_completo(consulta: str, max_pasos: int = 3) -> dict:
    try:
        chunks = recuperar(consulta)
        if chunks:
            contexto = "\n\n".join(f"[{i+1}] {c['texto']}" for i, c in enumerate(chunks))
            resp = converse_con_reintentos(
                modelId=MODEL_ID,
                messages=[{"role": "user", "content": [{"text": f"<evidencia>\n{contexto}\n</evidencia>\n\n<pregunta>\n{consulta}\n</pregunta>"}]}],
                toolConfig=TOOL_CONFIG,
            )
            texto = resp["output"]["message"]["content"][0]["text"] if resp["stopReason"] != "tool_use" else "(requeriría un segundo paso de tool use)"
            resultado = {"ruta": "RAG", "resultado": texto}
        else:
            ticket_id = crear_ticket_idempotente(consulta)
            resultado = {"ruta": "ticket_creado", "resultado": f"Ticket #{ticket_id}"}
    except Exception as e:
        ticket_id = crear_ticket_idempotente(consulta, contexto_recuperado=f"[error: {e}]")
        resultado = {"ruta": "fallback_error", "resultado": f"Ticket #{ticket_id} (por error técnico)"}

    registrar_accion(consulta, decision=resultado["ruta"], resultado=resultado["resultado"])
    return resultado


print(workflow_completo("¿Qué es la inyección de prompts?"))

## 11. Laboratorio: sistema de tickets integrado

**🏠 Tarea de la semana — entregar/completar antes de la Clase 10.**

**Checkpoint del proyecto integrador (Clase 9)**: sistema de tickets en SQLite funcionando sobre tu RAG real, con `consultar_estado_ticket` integrada por tool calling y contrato validado.

### Consigna

1. Conectá `workflow_completo` a tu RAG real (Clases 6-8, corpus de MDN) en vez del corpus de juguete de hoy.
2. Confirmá que `consultar_estado_ticket` tiene su contrato validado con Pydantic (sección 3) — ningún argumento del modelo llega a la base de datos sin validar.
3. Agregá al menos 2 preguntas de prueba que deberían crear un ticket (fuera de dominio) y 2 que deberían responderse con RAG.
4. Confirmá idempotencia: la misma pregunta fuera de dominio, preguntada dos veces, no debe crear dos tickets.
5. Revisá `registro_acciones` después de correr tus pruebas — tiene que poder explicar, para cada consulta, qué ruta tomó el workflow y por qué.

### Criterio de éxito

- El sistema nunca ejecuta `consultar_estado_ticket` con un argumento sin validar.
- Una pregunta fuera de dominio crea **un** ticket, no una alucinación de respuesta.
- La misma pregunta repetida no duplica el ticket.
- `registro_acciones` permite reconstruir, para cualquier consulta de prueba, la decisión que tomó el sistema.

Nota: `escalar_ticket` (la acción con aprobación humana) **no es parte de esta clase** — se agrega en la Clase 10, con LangGraph, como el workflow con estado completo.

In [ ]:
def tu_workflow_integrado(consulta: str) -> dict:
    # TODO: adaptar workflow_completo para usar tu índice real (Clase 5) y tu LLMProvider (Clase 3)
    # TODO: confirmar que usa crear_ticket_idempotente, no crear_ticket directo
    # TODO: confirmar que registra cada llamada con registrar_accion
    raise NotImplementedError


# Prueba (va a fallar hasta que completes el TODO):
# print(tu_workflow_integrado("una pregunta real sobre tu corpus"))

## Cierre

- Hoy el sistema dejó de limitarse a responder: ahora decide entre dos rutas (RAG o crear un ticket) y puede ejecutar una herramienta validada. Cada pieza (contrato, validación, límites, reintentos, idempotencia, registro) existe para que esa libertad no se vuelva impredecible.
- `escalar_ticket` (la acción que sí requiere aprobación humana) queda pendiente para la próxima clase — necesita el framework de estado de LangGraph para implementarse bien.
- Próxima clase: LangGraph, human-in-the-loop y MCP — el workflow de hoy se reescribe como un grafo explícito, y se agrega el punto de aprobación humana para `escalar_ticket`.
- Antes de la próxima clase: dejar el laboratorio de la sección 10 funcionando sobre tu corpus real.